In [10]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 1 - WEEK 12 BAYESIAN OPTIMISATION
# Run from inside week12/
# ============================================================
#
# Important:
# Function 1 is maximised on the ORIGINAL objective.
#
# We use only positive linear scaling:
#
#     Y_scaled = Y / max(abs(Y))
#
# This preserves ordering and argmax exactly.
#
# Because the objective scale is unusual, the GP is treated
# mainly as a LOCAL RANKING model rather than a reliable
# predictor of absolute objective magnitude.
# ============================================================


# ------------------------------------------------------------
# 1. Load Week 12 cumulative data
# ------------------------------------------------------------

X = np.load(
    "function1/initial_inputs.npy"
)

Y = np.load(
    "function1/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 11 calibration check
# ------------------------------------------------------------
#
# Week 11 selected:
# [0.698187, 0.734173]
#
# Previous GP prediction was approximately:
# raw mean = 2.11688e-06
# raw std  = 4.13798e-06
#
# Actual:
# 6.591260182136635e-12
#
# The z residual is useful as a diagnostic only.
# It does NOT mean the GP absolute scale is trustworthy.
# ------------------------------------------------------------

week11_pred_mean_raw = 2.11688e-06
week11_pred_std_raw = 4.13798e-06

week11_actual = (
    6.591260182136635e-12
)

week11_error = (
    week11_actual
    - week11_pred_mean_raw
)

week11_z_error = (
    week11_error
    / week11_pred_std_raw
)

print("\n================================")
print("WEEK 11 CALIBRATION CHECK")
print("================================")

print(
    "Predicted raw mean:",
    week11_pred_mean_raw
)

print(
    "Predicted raw std :",
    week11_pred_std_raw
)

print(
    "Actual            :",
    week11_actual
)

print(
    "\nPrediction error:",
    week11_error
)

print(
    "\nError / predicted std:",
    week11_z_error
)


# ------------------------------------------------------------
# 3. Positive linear scaling
# ------------------------------------------------------------

y_scale = np.max(
    np.abs(Y)
)

Y_scaled = (
    Y / y_scale
)

best_y_scaled = (
    best_y / y_scale
)

print("\n================================")
print("POSITIVE LINEAR SCALING")
print("================================")

print(
    "y_scale:",
    y_scale
)

print(
    "Original argmax:",
    np.argmax(Y)
)

print(
    "Scaled argmax:",
    np.argmax(Y_scaled)
)

print(
    "Ordering preserved:",
    np.argmax(Y)
    == np.argmax(Y_scaled)
)


# ------------------------------------------------------------
# 4. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(2) * 0.2,
        length_scale_bounds=(
            0.005,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-8,
        noise_level_bounds=(
            1e-10,
            1e-2
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(
    X,
    Y_scaled
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0
    / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)
print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask]
    - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# Slightly larger than nearest observed spacing,
# but still capped conservatively.

empirical_cap = min(
    1.25 * nearest_distance,
    0.05
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print(
    "Nearest observed point:",
    nearest_distance
)

print(
    "\nEmpirical cap:",
    empirical_cap
)


# ------------------------------------------------------------
# 7. ARD-informed trust region
# ------------------------------------------------------------
#
# x2 has historically had a much shorter lengthscale
# than x1, so its trust width should naturally be smaller.
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.22 * lengthscales,
    0.006,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x
    - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x
    + trust_half_width
)

print("\n================================")
print("WEEK 12 TRUST REGION")
print("================================")

print("Centre:")
print(
    best_x
)

print(
    "\nHalf-widths:"
)
print(
    trust_half_width
)

print(
    "\nLower:"
)
print(
    lower
)

print(
    "\nUpper:"
)
print(
    upper
)


# ------------------------------------------------------------
# 8. Dense trust-region candidates
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

candidates = rng.uniform(
    lower,
    upper,
    size=(
        400000,
        2
    )
)

tree = cKDTree(
    X
)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "\nCandidates after duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 9. GP predictions
# ------------------------------------------------------------

mu_scaled, sigma_scaled = gp.predict(
    candidates,
    return_std=True
)

mu_raw = (
    mu_scaled
    * y_scale
)

sigma_raw = (
    sigma_scaled
    * y_scale
)


# ------------------------------------------------------------
# 10. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu_scaled,
    sigma_scaled,
    best_y_scaled,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "scaled mean =",
    mu_scaled[ei_idx]
)

print(
    "scaled std =",
    sigma_scaled[ei_idx]
)

print(
    "raw mean =",
    mu_raw[ei_idx]
)

print(
    "raw std =",
    sigma_raw[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)


# ------------------------------------------------------------
# 11. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu_scaled
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "scaled mean =",
    mu_scaled[mean_idx]
)

print(
    "scaled std =",
    sigma_scaled[mean_idx]
)

print(
    "raw mean =",
    mu_raw[mean_idx]
)

print(
    "raw std =",
    sigma_raw[mean_idx]
)


# ------------------------------------------------------------
# 12. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta
        * sigma_scaled
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"beta={beta}",
        "\n candidate =",
        candidates[idx],
        "\n scaled mean =",
        round(
            mu_scaled[idx],
            8
        ),
        "\n scaled std =",
        round(
            sigma_scaled[idx],
            8
        ),
        "\n raw mean =",
        mu_raw[idx],
        "\n raw std =",
        sigma_raw[idx],
        "\n"
    )


# ------------------------------------------------------------
# 13. Distance from new incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta
        * sigma_scaled
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 14. Trust-region boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(
        len(x)
    ):

        if abs(
            x[j]
            - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j]
            - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(
        status
    )


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta
        * sigma_scaled
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )


# ------------------------------------------------------------
# 15. Axis-profile diagnostic
# ------------------------------------------------------------
#
# This checks each coordinate systematically while holding
# the other coordinate fixed at the current best.
#
# It helps us see whether the updated GP still prefers
# movement mainly in x1, without manually imposing that.
# ------------------------------------------------------------

print("\n================================")
print("AXIS PROFILE DIAGNOSTIC")
print("================================")

for j in range(2):

    values = np.linspace(
        lower[j],
        upper[j],
        20000
    )

    profile = np.tile(
        best_x,
        (
            len(values),
            1
        )
    )

    profile[:, j] = values

    profile_distance, _ = tree.query(
        profile,
        k=1
    )

    valid_profile = profile[
        profile_distance > 0.01
    ]

    if len(valid_profile) == 0:

        print(
            f"\nx{j+1}: no admissible points"
        )

        continue

    profile_mu, profile_sigma = gp.predict(
        valid_profile,
        return_std=True
    )

    idx = np.argmax(
        profile_mu
    )

    print(
        f"\nx{j+1} profile best:"
    )

    print(
        "candidate =",
        valid_profile[idx]
    )

    print(
        "raw mean =",
        profile_mu[idx]
        * y_scale
    )

    print(
        "raw std =",
        profile_sigma[idx]
        * y_scale
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            valid_profile[idx]
            - best_x
        )
    )

DATA
X shape: (21, 2)
Y shape: (21,)

Current best:
[0.698187 0.734173] -> 6.591260182136635e-12

Y range:
min = -0.0036060626443634764
max = 6.591260182136635e-12
std = 0.0007679429254738155

WEEK 11 CALIBRATION CHECK
Predicted raw mean: 2.11688e-06
Predicted raw std : 4.13798e-06
Actual            : 6.591260182136635e-12

Prediction error: -2.1168734087398177e-06

Error / predicted std: -0.5115716868471616

POSITIVE LINEAR SCALING
y_scale: 0.0036060626443634764
Original argmax: 20
Scaled argmax: 20
Ordering preserved: True


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
0.253**2 * Matern(length_scale=[2, 0.038], nu=2.5) + WhiteKernel(noise_level=1e-10)

ARD lengthscales:
[2.         0.03803489]

Normalised inverse-lengthscale sensitivity:
[0.01866253 0.98133747]

EMPIRICAL LOCAL SCALE
Nearest observed point: 0.03285757981819776

Empirical cap: 0.0410719747727472

WEEK 12 TRUST REGION
Centre:
[0.698187 0.734173]

Half-widths:
[0.04107197 0.00836768]

Lower:
[0.65711503 0.72580532]

Upper:
[0.73925897 0.74254068]

Candidates after duplicate filtering:
236372

PRIMARY EI
candidate = [0.65878691 0.74254012]
scaled mean = -5.781636815527236e-06
scaled std = 0.00588452961150612
raw mean = -2.084894454374937e-08
raw std = 2.121998241170294e-05
EI = 0.0023446970638034193

HIGHEST PREDICTED MEAN
candidate = [0.72107372 0.73401697]
scaled mean = 9.25799370818936e-05
scaled std = 0.00016279360871142816
raw mean = 3.3384905272853753e-07
raw std = 5.870439511154057e-07

UCB DIAGNOSTICS

beta=0.05 
 candidate = [0.65755633 0.74253314] 
 scal

In [11]:
# ============================================================
# FINAL FUNCTION 1 - WEEK 12 SELECTION
# ============================================================
#
# Week 11 produced a new incumbent:
# [0.698187, 0.734173]
#
# For Week 12:
# - EI and UCB were pulled toward the x2 upper boundary
#   mainly by uncertainty.
# - The highest posterior mean and the x1 axis profile
#   agreed on a move toward x1 ≈ 0.721.
# - Keeping x2 fixed at the successful Week 11 value gives
#   a more controlled exploitation move.
#
# Therefore select the axis-profile candidate.
# ============================================================

week12_candidate = np.array([
    0.72109193,
    0.734173
])

print("================================")
print("FUNCTION 1 - WEEK 12 FINAL")
print("================================")

print("\nCandidate:")
print(week12_candidate)

# GP prediction for final candidate
final_mu_scaled, final_sigma_scaled = gp.predict(
    week12_candidate.reshape(1, -1),
    return_std=True
)

final_mu_raw = (
    final_mu_scaled[0] * y_scale
)

final_sigma_raw = (
    final_sigma_scaled[0] * y_scale
)

print("\nPredicted raw mean:")
print(final_mu_raw)

print("\nPredicted raw std:")
print(final_sigma_raw)

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week12_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week12_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 1 - WEEK 12 FINAL

Candidate:
[0.72109193 0.734173  ]

Predicted raw mean:
3.245841149570769e-07

Predicted raw std:
6.234776900381271e-07

Distance from current best:
0.022904930000000046

Portal format:
0.721092-0.734173
